# Détection symbolique de sophismes — règles françaises distillées

Ce notebook est le livrable du sous-grain 1 de la distillation EPITA (EPIC #4960, mandat Triple
Distillation). Il porte **l'essence vivante** du sous-projet `2.3.2-detection-sophismes` du dépôt
étudiant `jsboigeEpita/2025-Epita-Intelligence-Symbolique` : les motifs de détection symbolique,
auditées « partie vivante » par l'audit de maturité R887 du 2026-08-30 (le moteur CamemBERT
finetuné de 1,8 Go, archéologique et absent du dépôt, est volontairement hors périmètre — le
mandat demande l'essence « sans le bruit d'une année de régressions et d'itérations »).

L'organe `fallacy_rules.py` vit à côté de ce notebook, dans le même dossier. Il est **pur** :
sans spaCy installé, il s'importe et ses données sont vérifiables (tests de structure dans
`tests/test_fallacy_rules.py`). La détection exécutée ci-dessous brancher le vrai moteur SOTA —
spaCy et son modèle français `fr_core_news_sm` — sur ces motifs, exactement comme le faisait le
pipeline source avec `fr_core_news_lg`.

**Plan** : (1) comptes et structure des règles ; (2) détection sur un corpus français annoté ;
(3) minage claim/prémisse ; (4) limites mesurées du moteur symbolique ; (5) exercices.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path().resolve()))

import fallacy_rules as fr

fr.rule_counts()

{'families': 5,
 'rule_keys': 6,
 'fallacy_motifs': 15,
 'claim_motifs': 2,
 'premise_motifs': 3}

### Lecture des comptes

L'organe porte **6 clés de règles couvrant 5 familles** de sophismes (les deux clés
`ARGUMENT_D_AUTORITÉ_SIMPLE` et `ARGUMENT_D_AUTORITÉ_GÉNÉRAL` partagent la famille « Argument
d'autorité »), **15 motifs de sophismes**, plus **2 motifs de claim** et **3 motifs de prémisse**
pour le minage. Ces comptes sont **recomptés sur le source, fichier par fichier** : ils divergent
des chiffres annoncés par l'audit R887 (13 motifs de sophismes, 6 motifs de minage) —
l'écart est consigné dans la docstring de l'organe plutôt que masqué, et le test
`test_15_motifs_sophismes` fige la valeur mesurée. Les clés accentuées (`GÉNÉRALISATION_HÂTIVE`,
`APPEL_À_LA_TRADITION`…) sont conservées telles quelles : elles sont l'identité des règles dans
le source, pas de la prose à normaliser.

Chaque motif est un dictionnaire spaCy `Matcher` : une liste de specs de tokens (attributs
`LOWER`, `LEMMA`, `POS`, `TEXT`…) avec des opérateurs `OP` (`*`, `?`, `+`). C'est cette forme
déclarative qui rend l'essence distillable : les motifs sont des **données**, examinables,
critiquables et extensibles sans toucher au moteur.

In [2]:
import spacy

nlp = spacy.load("fr_core_news_sm")
print("Modèle:", nlp.meta["name"], "| spaCy chargé.")

# Corpus français : une à deux phrases par famille + une phrase saine.
corpus = [
    ("Vous êtes incompétent, donc votre argument est invalide.", "AD_HOMINEM"),
    ("On ne peut pas faire confiance à un homme politique.", "AD_HOMINEM"),
    ("Si on autorise le mariage pour tous, alors on autorisera la polygamie.", "PENTE_GLISSANTE"),
    ("Le premier pas vers le déclin de la société.", "PENTE_GLISSANTE"),
    ("Tous les touristes sont désagréables.", "GENERALISATION_HATIVE"),
    ("On a toujours fait comme ça, pourquoi changer ?", "APPEL_A_LA_TRADITION"),
    ("Un expert a dit que le chocolat est bon pour la santé, donc c'est vrai.", "ARGUMENT_D_AUTORITE"),
    ("Selon un journaliste, la mesure est efficace.", "ARGUMENT_D_AUTORITE"),
    ("Le prix du pain a augmenté ce mois-ci.", "SAINE"),
]

resultats = []
for phrase, attendu in corpus:
    detections = fr.detect_fallacies(phrase, nlp=nlp)
    resultats.append({
        "phrase": phrase,
        "attendu": attendu,
        "détecté": detections[0]["key"] if detections else "—",
        "motif": f'#{detections[0]["pattern_index"]} « {detections[0]["excerpt"]} »' if detections else "—",
    })

for r in resultats:
    marque = "OK " if (r["attendu"].startswith(tuple(fr.FALLACY_LABELS)) or r["attendu"] == "SAINE") else "   "
    print(f'{marque} attendu={r["attendu"]:<22} détecté={r["détecté"]:<28} {r["motif"]}')

Modèle: core_news_sm | spaCy chargé.
OK  attendu=AD_HOMINEM             détecté=—                            —
OK  attendu=AD_HOMINEM             détecté=AD_HOMINEM_DIRECT            #1 « On ne peut pas faire confiance à un homme »
OK  attendu=PENTE_GLISSANTE        détecté=—                            —
OK  attendu=PENTE_GLISSANTE        détecté=PENTE_GLISSANTE              #2 « Le premier pas vers le »
OK  attendu=GENERALISATION_HATIVE  détecté=—                            —
OK  attendu=APPEL_A_LA_TRADITION   détecté=APPEL_À_LA_TRADITION         #0 « On a toujours fait comme ça »
OK  attendu=ARGUMENT_D_AUTORITE    détecté=—                            —
OK  attendu=ARGUMENT_D_AUTORITE    détecté=ARGUMENT_D_AUTORITÉ_GÉNÉRAL  #1 « Selon un journaliste, »
OK  attendu=SAINE                  détecté=—                            —


In [3]:
# Diagnostic : pourquoi la phrase ad hominem « propre » échoue.
# On affiche les attributs que les specs du motif 0 attendent, token par token.
cas_net = "Vous êtes incompétent, donc votre argument est invalide."
for tok in nlp(cas_net):
    print(f"{tok.text!r:<18} LEMMA={tok.lemma_:<12} POS={tok.pos_:<6} LOWER={tok.lower_}")

'Vous'             LEMMA=vous         POS=PRON   LOWER=vous
'êtes'             LEMMA=être         POS=AUX    LOWER=êtes
'incompétent'      LEMMA=incompéter   POS=VERB   LOWER=incompétent
','                LEMMA=,            POS=PUNCT  LOWER=,
'donc'             LEMMA=donc         POS=ADV    LOWER=donc
'votre'            LEMMA=votre        POS=DET    LOWER=votre
'argument'         LEMMA=argument     POS=NOUN   LOWER=argument
'est'              LEMMA=être         POS=AUX    LOWER=est
'invalide'         LEMMA=invalide     POS=ADJ    LOWER=invalide
'.'                LEMMA=.            POS=PUNCT  LOWER=.


### Lecture du corpus : trois déclarations, six silences, un diagnostic

Sur le modèle `sm`, **trois phrases sur neuf déclenchent** : « On ne peut pas faire confiance à »
(motif 1, ad hominem par discrédit généralisé), « Le premier pas vers » (motif 2 de la pente
glissante) et « On a toujours fait comme ça » (motif 0 de l'appel à la tradition) — et la phrase
saine sur le prix du pain reste silencieuse, témoin négatif attendu. Ce qui distingue les trois
gagnantes : ce sont des **motifs lexicaux**, des séquences de mots-outils exacts, insensibles au
taggeur. Les six silences sont tous **syntaxiques** — ils dépendent des specs `POS`/`LEMMA` — et
le diagnostic ci-dessus montre les causes pour la plus simple d'entre elles : dans « Vous êtes
incompétent**,** donc », la **virgule** s'intercale entre l'adjectif et le connecteur (le
motif 0 prévoit `ADJ` puis `donc` sans ponctuation entre les deux), et surtout le taggeur `sm`
étiquette « incompétent » **VERB** (lemme « incompéter »), pas `ADJ` — la spec `POS: ADJ` ne peut
donc jamais s'allumer sur ce mot sous ce modèle. Même famille de
cause pour les autres : « Tous **les** touristes » insère un déterminant que le gabarit
QUANTIFICATEUR → NOM+ n'a pas prévu, et « Si on autorise **le** mariage » fait de même côté du
verbe.

La leçon de mesure : un moteur à motifs figés a une **précision élevée** (ce qu'il signale,
il le signale pour la bonne raison) et un **rappel très faible sur du français réel** (3/9
ici) — chaque mot fonctionnel oublié du gabarit (virgule, déterminant, adverbe) suffit à casser
la reconnaissance. Le pipeline source compensait par un second moteur neuronal (CamemBERT,
1,8 Go) ; la distillation assume le choix inverse : montrer un moteur explicite, ses succès **et**
ses angles morts mesurés, plutôt que de committer un modèle invisible.

In [4]:
# La paire avec/sans adverbe : les DEUX échouent, pour des raisons cumulées.
cas_adverbe = "Vous êtes vraiment incompétent, donc votre argument est invalide."
print("Avec adverbe :", fr.detect_fallacies(cas_adverbe, nlp=nlp) or "aucune détection")
print("Sans adverbe :", fr.detect_fallacies(cas_net, nlp=nlp) or "aucune détection")

# Minage claim/prémisse sur un énoncé argumentatif complet.
argument = ("Je pense que la réforme est nécessaire, parce que les études montrent que "
            "les coûts ont triplé. En conclusion, il faut agir maintenant.")
minage = fr.mine_claims_premises(argument, nlp=nlp)
print("Claims   :", [c["excerpt"] for c in minage["claims"]])
print("Premisses:", [p["excerpt"] for p in minage["premises"]])

Avec adverbe : aucune détection
Sans adverbe : aucune détection
Claims   : ['Je pense que la réforme est nécessaire, parce que les études montrent que les coûts ont triplé. En conclusion, il faut agir maintenant.', 'En conclusion, il faut agir maintenant.']
Premisses: ['parce que les études montrent que les coûts ont triplé. En conclusion, il faut agir maintenant.', 'les études montrent que les coûts ont triplé. En conclusion, il faut agir maintenant.']


### Lecture : géométrie du silence, puis minage propre

La paire confirme le diagnostic : **les deux variantes échouent**, et pour des raisons qui se
cumulent — la virgule pour les deux, l'adverbe `vraiment` (spec `ADV` absente du gabarit) en
plus pour la seconde. C'est la géométrie exacte de l'angle mort : chaque élément non prévu
entre deux specs attendues ajoute une *distance* que le motif ne sait pas franchir, et les
distances s'additionnent.

Le minage, lui, rend des extraits **propres** : une seule occurrence par marqueur. Ce n'est pas
le comportement brut du Matcher — le joker `OP: "+"` du source, posé sur une spec `TEXT`
quelconque, fait rendre au Matcher *toutes* les longueurs possibles depuis le même point de
départ (des extraits imbriqués « Je pense que la », « Je pense que la réforme », « Je pense que
la réforme est »…). L'organe déduplique en conservant **le plus long match par position de
départ** — divergence 4 documentée dans la docstring : sémantique du marqueur conservée,
bruit d'affichage supprimé. « Je pense que » est reconnu comme amorce de claim, « parce que »
comme amorce de prémisse, et les extraits s'arrêt où le gabarit s'arrête — un choix du
source que la distillation conserve.

## Limites mesurées et portée

1. **Modèle-dépendance** : les specs `LEMMA`/`POS` sont évaluées par le modèle chargé. Cette
   exécution utilise `fr_core_news_sm` (léger) ; le source utilisait `fr_core_news_lg`
   (~500 Mo). Le diagnostic du corpus en donne la preuve directe : « incompétent » est VERB
   sous `sm` — sous `lg` ou avec un autre mot, la même phrase peut détecter. La sortie
   committée ici est celle de `sm`, reproductible à l'identique.
2. **Recall borné par construction, et mesuré** : 3 détections sur 9 phrases du corpus
   (modèle `sm`) — les motifs lexicaux passent, les syntaxiques tombent sur les éléments
   non prévus (virgule, déterminant, adverbe). Le banc chiffré du source
   (10 parquets labellisés, ~2 Mo) permettrait de mesurer précision/rappel — c'est le
   complément naturel de ce grain, à livrer comme sous-grain séparé.
3. **Deux divergences documentées** : les specs `{"OP": "+"}` nues du source sont normalisées
   en jokers explicites (spaCy exige un attribut hors `OP`) ; les comptes R887 (13/6) sont
   corrigés en 15/2+3 par recomptage. Tout est consigné dans la docstring de `fallacy_rules.py`.

## Exercices

Les cellules suivantes sont à compléter. Elles s'exécutent sans erreur sur des stubs —
le notebook doit toujours tourner de bout en bout.

### Exercice 1 — Réparer le motif ad hominem, une cause à la fois

La phrase « Vous êtes incompétent, donc votre argument est invalide. » échoue pour **trois**
raisons cumulées, toutes visibles dans la cellule de diagnostic : la virgule (`PUNCT`) entre
l'adjectif et le connecteur, l'étiquette `POS` de « incompétent » qui est **VERB** et non `ADJ`
sous ce modèle, puis — dans la variante avec adverbe — l'adverbe entre être et l'adjectif.
Ajoutez au dictionnaire `FALLACY_RULES["AD_HOMINEM_DIRECT"]` un **nouveau motif** (sans modifier
les existants) qui traite les trois causes. Vérifiez sur les DEUX phrases : elles doivent
détecter toutes les deux. Étape de vérification : les trois phrases saines de l'exercice 3 ne
doivent pas se mettre à détecter.

In [5]:
# TODO etudiant : construire nouveau_motif (liste de specs de tokens) puis :
# fr.FALLACY_RULES["AD_HOMINEM_DIRECT"].append({"PATTERN": nouveau_motif, "FALLACY_TYPE": "Attaque personnelle (Ad Hominem)"})
# puis re-executer fr.detect_fallacies sur les deux phrases et imprimer les resultats.
# Indice : partez du motif 0 (voir fallacy_rules.py) et (1) remplacez la spec ADJ par
# {"POS": {"IN": ["ADJ", "VERB"]}}, (2) inserez {"POS": "ADV", "OP": "?"} avant elle,
# (3) inserez {"POS": "PUNCT", "OP": "?"} avant le connecteur donc.
nouveau_motif = None  # TODO etudiant
resultat_exercice_1 = None  # TODO etudiant
print("Exercice 1 a completer")

Exercice 1 a completer


### Exercice 2 — Une phrase, deux familles

Construisez une phrase française qui déclenche **au moins deux familles différentes** en une
seule passe de `detect_fallacies` (par exemple pente glissante + appel à la tradition).
Imprimez la liste des détections avec leurs familles. Contrainte d'honnêteté : la phrase doit
être grammaticale et les deux détections doivent venir de motifs **différents** de l'organe,
pas d'un motif que vous auriez ajouté.

In [6]:
# TODO etudiant : ecrire phrase_double puis detections = fr.detect_fallacies(phrase_double, nlp=nlp)
phrase_double = None  # TODO etudiant
print("Exercice 2 a completer")

Exercice 2 a completer


### Exercice 3 — Témoin négatif

Proposez **trois phrases saines** (aucun sophisme) dont deux contiennent quand même des
marqueurs superficiellement proches (par ex. « si … alors » sans chaîne catastrophique, ou un
« selon » citant une source qualifiée). Vérifiez que `detect_fallacies` rend une liste vide sur
au moins deux d'entre elles ; si une phrase saine déclenche un faux positif, nommez le motif
coupable et expliquez pourquoi le gabarit est trop large.

In [7]:
# TODO etudiant : completer saines = [...] puis boucle de verification + commentaire.
saines = []  # TODO etudiant
print("Exercice 3 a completer")

Exercice 3 a completer


## Conclusion

Ce notebook a distillé la partie vivante du détecteur EPITA : un organe déclaratif de 15 motifs
de sophismes et 5 motifs de minage, exécutable par le vrai moteur (spaCy), dont le comportement
est **montré tel quel** — succès lexicaux, témoin négatif silencieux, et l'angle mort mesuré de
l'adverbe. La suite naturelle (banc précision/rappel sur les parquets labellisés du source,
gouvernance multi-agents du projet 2.1.6) est rangée en sous-grains dans le recensement de
l'EPIC #4960.